# DuckSeg example: protein recruitment to laser-induced DNA damage

<a href="https://colab.research.google.com/github/ronalabrcns/DuckSeg/blob/main/notebooks/DuckSeg_Example.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

This notebook runs the full [DuckSeg](https://github.com/ronalabrcns/DuckSeg) pipeline on an example dataset:

1. **Setup**: install DuckSeg and CellSAM
2. **Data**: download the example videos (or use your own from Google Drive)
3. **Preprocessing**: preview the cell masks on a few frames
4. **Segment and track** every cell that touches the laser stripe
5. **Review** the cells and remove failures
6. **Plot** the recruitment curve (mean ± SEM)
7. **Export** the per-cell values as CSV

Documentation: [duckseg.readthedocs.io](https://duckseg.readthedocs.io/en/latest/workflow.html)

**Before you start**

* *Runtime → Change runtime type → GPU* (T4 is enough).
* A free DeepCell access token from [users.deepcell.org](https://users.deepcell.org/). In Colab click the 🔑 *Secrets* icon on the left, add a secret named `DEEPCELL_ACCESS_TOKEN` and switch on *Notebook access*.

## 1. Setup

In [ ]:
# Install DuckSeg and CellSAM (about 2-3 minutes)
![ -d DuckSeg ] || git clone --depth 1 https://github.com/ronalabrcns/DuckSeg.git
!pip install --quiet -e ./DuckSeg
!pip install --quiet git+https://github.com/vanvalenlab/cellSAM.git@362a8e521bcd7313d5cb280bcd4cfc12977c1e9f
!pip install --quiet ipympl

In [ ]:
# Restart the runtime so the new packages are used.
# Colab shows "Your session crashed" or reconnects: that is expected. Continue with the next cell.
import os
os.kill(os.getpid(), 9)

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    sys.path.insert(0, "/content/DuckSeg/src")   # in case the editable install is not picked up yet
    from google.colab import output, userdata
    output.enable_custom_widget_manager()        # needed for the interactive widgets
    try:
        os.environ["DEEPCELL_ACCESS_TOKEN"] = userdata.get("DEEPCELL_ACCESS_TOKEN")
    except Exception:
        raise RuntimeError("Add your DeepCell token to Colab Secrets as DEEPCELL_ACCESS_TOKEN "
                           "and enable notebook access (see the top of this notebook).")

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (this will be slow!)")

## 2. Data

`DATA_SOURCE = "example"` downloads the example dataset. To analyse your own recordings, set it to `"drive"` and point `DRIVE_DIR` to a Google Drive folder with one video and one ROI mask per experiment:

```
MSH2_001.nd2        # video (.nd2, .tif or .tiff)
MSH2_001_ROI.tif    # laser stripe mask, same size as the video
```

See [Prepare your data](https://duckseg.readthedocs.io/en/latest/guides/data_preparation.html).

In [ ]:
DATA_SOURCE = "example"        # "example" or "drive"

# Example dataset: a .zip with the videos and *_ROI.tif masks
EXAMPLE_DATA_URL = ""          # TODO: Zenodo / GitHub release link of the example dataset
EXAMPLE_NAME = "MSH2"          # condition name, used as plot label

# Your own data on Google Drive
DRIVE_DIR = "/content/drive/MyDrive/DuckSeg/my_condition"
DRIVE_NAME = "my_condition"

In [ ]:
import urllib.request, zipfile, glob

if DATA_SOURCE == "example":
    if not EXAMPLE_DATA_URL:
        raise ValueError("EXAMPLE_DATA_URL is not set yet. Use DATA_SOURCE = 'drive' with your own data.")
    data_root = "/content/example_data" if IN_COLAB else os.path.abspath("example_data")
    if not os.path.isdir(data_root):
        os.makedirs(data_root)
        print("Downloading example data ...")
        zip_path, _ = urllib.request.urlretrieve(EXAMPLE_DATA_URL, os.path.join(data_root, "data.zip"))
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(data_root)
        os.remove(zip_path)
    # the folder that contains the *_ROI.tif files
    data_dir = os.path.dirname(sorted(glob.glob(os.path.join(data_root, "**", "*_ROI.tif"), recursive=True))[0])
    name = EXAMPLE_NAME
elif DATA_SOURCE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    data_dir, name = DRIVE_DIR, DRIVE_NAME
else:
    raise ValueError("DATA_SOURCE must be 'example' or 'drive'")

print("Data folder:", data_dir)

In [ ]:
from DuckSeg.experiment_loader import ExperimentBatch

batch = ExperimentBatch(name, data_dir)
experiments = batch.experiments()
print(f"{len(experiments)} experiments:", [e.name for e in experiments])

# Cut all videos to the shortest one so that the time axis is the same everywhere
batch.max_len = min(e.frame_count() for e in experiments)
print("Frames per video:", batch.max_len)

### Check the laser stripe

The detected stripe is blacked out in the image below. It should sit on the irradiated line. If it does not, use the optional ROI editor.

In [ ]:
for experiment in batch.experiments()[:2]:
    experiment.plot_frame_with_roi_masked()

In [ ]:
# Optional: move the stripe interactively (▲ / ▼ buttons, saved automatically)
EDIT_ROI = False

if EDIT_ROI:
    %matplotlib widget
    from DuckSeg.roi_editor import show_roi_editors
    show_roi_editors(batch.experiments())

## 3. Preprocessing

The transforms are applied to every frame before CellSAM. Tune them until most nuclei are outlined; the preview only processes a few frames, so it is quick. Details: [Tune the preprocessing](https://duckseg.readthedocs.io/en/latest/guides/preprocessing.html).

In [ ]:
%matplotlib inline
from DuckSeg.image_transforms import convert_to_u8, set_brightness, threshold_image, adjust_contrast
from DuckSeg.image_transformer import generate_masks_using_filters_and_transforms

transforms = [
    convert_to_u8,        # required: CellSAM needs 8-bit images
    set_brightness(20),   # scale every frame to the same mean brightness
    threshold_image(20),  # binary image: strong, clean signal for the nuclei
]

generate_masks_using_filters_and_transforms(
    batch,
    experiment_filter=[0],    # preview the first experiment
    frame_filter=2,           # first and last frame
    transforms=transforms,
    show_after_transform=[],  # e.g. [0, 1] to also show intermediate steps
)

## 4. Segment and track

CellSAM segments every frame, and each cell touching the stripe is followed through the video. This is the slow step (a few minutes on a GPU); results are cached next to the videos, so re-running is instant. After changing `transforms`, run once with `use_cache=False`.

In [ ]:
from DuckSeg.experiment_evaluator import evaluate_batch

vids = evaluate_batch(batch, transforms=transforms, use_cache=True)
print(f"{len(vids)} tracked cells")

## 5. Review the cells

Automatic filters remove cells with implausible mask areas. Then look at the cells: tick a checkbox to exclude a cell (red border = already excluded). Details: [Review and filter cells](https://duckseg.readthedocs.io/en/latest/guides/outlier_filtering.html).

In [ ]:
from DuckSeg.outlier_filters import outlier_vids, area_between, max_area_change_between_frames

range_modifier = 4                 # width of the scored band: 4 x the drawn stripe

filters = [
    area_between(10, 5000),                  # mask area in pixels, in every frame
    max_area_change_between_frames(0.3),     # no jumps > 30 % between frames
]
extra_manual_outliers = set()      # cell ids you always want to exclude, e.g. {"MSH2_001_3"}

outliers = outlier_vids(vids, extra_manual_outliers, filters)
print(f"{len(outliers)} of {len(vids)} cells removed by the filters")

In [ ]:
from DuckSeg.cell_video_visualizer import normalize_videos_for_display, show_video_grid_jupyter

manual_outliers = set()            # filled by the checkboxes below
display_vids = normalize_videos_for_display(vids, range_modifier=range_modifier)

for i in range(0, len(display_vids), 30):          # 30 cells per grid
    update = show_video_grid_jupyter(display_vids[i:i + 30], manual_outliers=manual_outliers)
    update(outliers, extra_manual_outliers)

In [ ]:
cell_filter = lambda vid: vid.id not in outliers and vid.id not in manual_outliers
kept = [vid for vid in vids if cell_filter(vid)]
print(f"Analysing {len(kept)} of {len(vids)} cells")

## 6. Recruitment curve

Mean over the kept cells with the standard error of the mean as a band. `Ratio` = inside / outside the band, `Diff` = inside − outside. With `baseline_correct=True` every cell starts at zero. See [Concepts](https://duckseg.readthedocs.io/en/latest/concepts.html).

In [ ]:
from DuckSeg.experiment_evaluator import ScoreType, plot_brightness_score

os.makedirs("output", exist_ok=True)

for score_type in [ScoreType.Ratio, ScoreType.Diff]:
    plot_brightness_score(batch, score_type=score_type, range_modifier=range_modifier,
                          cell_filter=cell_filter, baseline_correct=True,
                          save_path=f"output/recruitment_{score_type.name.lower()}.png")

In [ ]:
# Optional: every cell's trace in 3D, sorted by its mean (unusual traces stand out)
from DuckSeg.experiment_evaluator import calculate_brightness_score_for_videos, plot_all_vid_scores

ids, scores = calculate_brightness_score_for_videos(vids, score_type=ScoreType.Ratio,
                                                    range_modifier=range_modifier, cell_filter=cell_filter)
plot_all_vid_scores(scores, ids)

## 7. Export

One row per frame and four columns per cell: raw `_normal` and `_ROI` intensities, and baseline-corrected `_diff` and `_ratio` scores (as in the plots). See [Export results](https://duckseg.readthedocs.io/en/latest/guides/export_results.html).

In [ ]:
from DuckSeg.video_exporter import export_cell_videos

csv_path = f"output/{name}_brightness_scores.csv"
export_cell_videos(kept, csv_path, range_modifier=range_modifier, baseline_correct=True)
print("Saved", csv_path)

if IN_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive("duckseg_results", "zip", "output")
    files.download("duckseg_results.zip")

## Next steps

* Analyse your own data: set `DATA_SOURCE = "drive"` in section 2.
* Compare conditions (one folder per condition) with `plot_brightness_scores`: [Compare conditions](https://duckseg.readthedocs.io/en/latest/guides/compare_conditions.html).
* Problems? See [Troubleshooting](https://duckseg.readthedocs.io/en/latest/troubleshooting.html).